In [ ]:
!pip install -q flask pyngrok openai openai-whisper

import os
import json
import time
import uuid
import tempfile
import threading
import traceback
import getpass
from typing import Any, Dict, List

import whisper
from flask import Flask, request, jsonify
from pyngrok import ngrok
from openai import OpenAI


# =========================
# CONFIG
# =========================

GROQ_API_KEY = os.environ.get("GROQ_API_KEY") or getpass.getpass("Enter your Groq API key: ")
NGROK_AUTH_TOKEN = os.environ.get("NGROK_AUTH_TOKEN") or getpass.getpass("Enter your ngrok auth token: ")

GROQ_MODEL = "llama-3.1-8b-instant"
WHISPER_MODEL_SIZE = "base"
MAX_HISTORY_TURNS = 8
TEMP_DIR = tempfile.gettempdir()

ROLE_LIBRARY = {
    "Software Engineer Intern": {
        "skills": ["programming fundamentals", "projects", "debugging", "DSA basics", "teamwork"],
        "question_mix": ["intro", "project", "technical", "behavioral", "situational"],
    },
    "Frontend Developer": {
        "skills": ["HTML", "CSS", "JavaScript", "React", "accessibility", "UI performance"],
        "question_mix": ["intro", "project", "technical", "debugging", "behavioral"],
    },
    "Backend Developer": {
        "skills": ["APIs", "databases", "authentication", "system design basics", "debugging"],
        "question_mix": ["intro", "project", "technical", "system_design", "behavioral"],
    },
    "Data Analyst": {
        "skills": ["SQL", "Excel", "Python", "data cleaning", "dashboards", "business insights"],
        "question_mix": ["intro", "project", "case", "technical", "behavioral"],
    },
    "HR Round": {
        "skills": ["communication", "motivation", "self-awareness", "teamwork", "career goals"],
        "question_mix": ["intro", "behavioral", "situational", "closing"],
    },
}

FALLBACK_QUESTIONS = [
    "Tell me about a project you are proud of and your exact contribution.",
    "Describe a challenge you faced and how you solved it.",
    "Can you give me a specific example where you learned something quickly?",
    "Why are you interested in this role?",
    "What would you improve if you repeated your last project?",
]


# =========================
# LOAD MODELS ONCE
# =========================

print("Loading Whisper model. This can take a minute...")
whisper_model = whisper.load_model(WHISPER_MODEL_SIZE)
whisper_lock = threading.Lock()
print("Whisper loaded.")

client = OpenAI(api_key=GROQ_API_KEY, base_url="https://api.groq.com/openai/v1")
app = Flask(__name__)


# =========================
# HELPERS
# =========================

def clamp(value, minimum, maximum):
    return max(minimum, min(value, maximum))


def safe_json_loads(text: str, fallback: Dict[str, Any]) -> Dict[str, Any]:
    try:
        return json.loads(text)
    except Exception:
        pass

    try:
        start = text.find("{")
        end = text.rfind("}") + 1
        if start >= 0 and end > start:
            return json.loads(text[start:end])
    except Exception:
        pass

    fallback["raw_response"] = text
    return fallback


def normalize_list(value, fallback):
    if isinstance(value, list):
        return [str(item).strip() for item in value if str(item).strip()][:5]
    if isinstance(value, str) and value.strip():
        return [value.strip()]
    return fallback


def transcribe_audio_file(audio_path: str) -> str:
    with whisper_lock:
        result = whisper_model.transcribe(
            audio_path,
            language="en",
            fp16=False,
            condition_on_previous_text=False,
            no_speech_threshold=0.6,
        )
    return result.get("text", "").strip()


def count_fillers(text: str) -> int:
    lower = text.lower()
    single_fillers = {"um", "uh", "like", "actually", "basically", "right", "literally", "okay"}
    phrase_fillers = ["you know", "i mean", "sort of", "kind of"]
    words = [word.strip(".,!?;:\"'()[]{}").lower() for word in lower.split()]
    return sum(1 for word in words if word in single_fillers) + sum(lower.count(p) for p in phrase_fillers)


def estimate_wpm(text: str, duration_seconds: float) -> int:
    if duration_seconds <= 0:
        return 0
    return int(len(text.split()) / (duration_seconds / 60))


def local_speech_score(transcript: str, duration_seconds: float = 0) -> Dict[str, Any]:
    words = len(transcript.split())
    fillers = count_fillers(transcript)
    wpm = estimate_wpm(transcript, duration_seconds)
    confidence = 70

    if words < 12:
        confidence -= 28
    elif words < 35:
        confidence -= 10
    elif words <= 160:
        confidence += 10
    else:
        confidence -= 8

    confidence -= fillers * 3

    if wpm:
        if 110 <= wpm <= 170:
            confidence += 10
        elif wpm < 90 or wpm > 210:
            confidence -= 12

    return {
        "confidence": int(clamp(confidence, 0, 100)),
        "filler_count": int(fillers),
        "wpm_estimate": int(wpm),
        "word_count": int(words),
    }


def star_breakdown(transcript: str) -> Dict[str, str]:
    lower = transcript.lower()
    has_situation = any(x in lower for x in ["when", "during", "in my project", "at", "while"])
    has_task = any(x in lower for x in ["needed to", "had to", "responsible", "goal", "task"])
    has_action = any(x in lower for x in ["i built", "i created", "i implemented", "i solved", "i used", "i did"])
    has_result = any(x in lower for x in ["result", "improved", "reduced", "increased", "completed", "learned", "%"])
    return {
        "situation": "present" if has_situation else "missing",
        "task": "present" if has_task else "weak_or_missing",
        "action": "present" if has_action else "missing",
        "result": "present" if has_result else "missing",
    }


def build_conversation(history: List[Dict[str, Any]]) -> str:
    lines = []
    for index, item in enumerate(history[-MAX_HISTORY_TURNS:], start=1):
        lines.append(f"Turn {index}")
        lines.append(f"Question: {item.get('question', '')}")
        lines.append(f"Answer: {item.get('answer', '')}")
        if item.get("score") is not None:
            lines.append(f"Score: {item.get('score')}/10")
        if item.get("feedback"):
            lines.append(f"Feedback: {item.get('feedback')}")
        lines.append("")
    return "\n".join(lines).strip()


def default_turn(transcript: str) -> Dict[str, Any]:
    return {
        "score": 6,
        "breakdown": {
            "relevance": 6,
            "clarity": 6,
            "specificity": 5,
            "structure": 5,
            "technical_depth": 5,
            "impact": 4,
        },
        "feedback": "The answer is understandable, but it needs a clearer structure, stronger examples, and measurable impact.",
        "strengths": ["The candidate attempted to answer the question."],
        "weaknesses": ["The answer could be more specific and structured."],
        "suggestions": ["Use the STAR method and include a concrete result."],
        "improved_sample_answer": "A stronger answer would briefly explain the situation, your responsibility, the specific actions you took, and the measurable result.",
        "next_question": "Can you give me a specific example from a project where you solved a difficult problem?",
        "difficulty": "medium",
        "question_type": "follow_up",
        "ideal_answer_hint": "Include context, your exact contribution, tools used, and the outcome.",
        "red_flags": [],
    }


def ai_interview_turn(
    history: List[Dict[str, Any]],
    transcript: str,
    persona: str,
    role: str,
    interview_mode: str,
    time_remaining_seconds: int,
    speech_metrics: Dict[str, Any],
    resume_context: str = "",
    job_description: str = "",
) -> Dict[str, Any]:
    role_profile = ROLE_LIBRARY.get(role, ROLE_LIBRARY["Software Engineer Intern"])
    conversation = build_conversation(history)
    star = star_breakdown(transcript)

    system_prompt = f"""
You are a realistic AI interview coach and interviewer.

Role: {role}
Role skills: {", ".join(role_profile["skills"])}
Interview mode: {interview_mode}
Persona: {persona}
Time remaining: {time_remaining_seconds} seconds

Rules:
- Ask exactly one next question.
- If the answer is vague, ask a targeted follow-up.
- If the answer is strong, increase difficulty gradually.
- If the answer is off-topic, politely redirect.
- If time is below 60 seconds, ask a closing question.
- Be realistic, concise, and useful. Do not flatter unnecessarily.
- Do not invent candidate achievements.
- If resume or job description context exists, use it naturally.

Return ONLY valid JSON. No markdown. No extra text.

Required JSON schema:
{{
  "score": 0,
  "breakdown": {{
    "relevance": 0,
    "clarity": 0,
    "specificity": 0,
    "structure": 0,
    "technical_depth": 0,
    "impact": 0
  }},
  "feedback": "short realistic feedback",
  "strengths": ["strength 1", "strength 2"],
  "weaknesses": ["weakness 1", "weakness 2"],
  "suggestions": ["suggestion 1", "suggestion 2"],
  "improved_sample_answer": "short improved answer or answer pattern",
  "next_question": "one next interview question only",
  "difficulty": "easy|medium|hard",
  "question_type": "intro|behavioral|technical|follow_up|situational|project|system_design|closing",
  "ideal_answer_hint": "short hint for what a strong answer should include",
  "red_flags": ["optional concern"]
}}
"""

    user_prompt = f"""
Conversation:
{conversation if conversation else "No previous conversation."}

Latest candidate answer:
{transcript}

Speech metrics:
{json.dumps(speech_metrics)}

STAR heuristic:
{json.dumps(star)}

Resume context:
{resume_context[:2500] if resume_context else "Not provided."}

Job description:
{job_description[:2500] if job_description else "Not provided."}

Now evaluate the latest answer and choose the next best question.
"""

    response = client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0.42,
        max_tokens=900,
    )

    parsed = safe_json_loads(response.choices[0].message.content, default_turn(transcript))
    fallback = default_turn(transcript)

    parsed["score"] = int(clamp(int(parsed.get("score", fallback["score"])), 0, 10))
    parsed["breakdown"] = parsed.get("breakdown", fallback["breakdown"])
    for key in fallback["breakdown"]:
        parsed["breakdown"][key] = int(clamp(int(parsed["breakdown"].get(key, 5)), 0, 10))

    parsed["feedback"] = str(parsed.get("feedback", fallback["feedback"])).strip()
    parsed["strengths"] = normalize_list(parsed.get("strengths"), fallback["strengths"])
    parsed["weaknesses"] = normalize_list(parsed.get("weaknesses"), fallback["weaknesses"])
    parsed["suggestions"] = normalize_list(parsed.get("suggestions"), fallback["suggestions"])
    parsed["red_flags"] = normalize_list(parsed.get("red_flags"), [])
    parsed["improved_sample_answer"] = str(parsed.get("improved_sample_answer", fallback["improved_sample_answer"])).strip()
    parsed["next_question"] = str(parsed.get("next_question", fallback["next_question"])).strip() or fallback["next_question"]
    parsed["difficulty"] = str(parsed.get("difficulty", "medium")).strip().lower()
    parsed["question_type"] = str(parsed.get("question_type", "follow_up")).strip().lower()
    parsed["ideal_answer_hint"] = str(parsed.get("ideal_answer_hint", fallback["ideal_answer_hint"])).strip()
    parsed["star"] = star

    return parsed


def generate_opening_question(role: str, persona: str, interview_mode: str, resume_context: str = "", job_description: str = "") -> Dict[str, Any]:
    prompt = f"""
Create the first question for a mock interview.
Role: {role}
Persona: {persona}
Mode: {interview_mode}
Resume context: {resume_context[:1800] if resume_context else "Not provided"}
Job description: {job_description[:1800] if job_description else "Not provided"}

Return ONLY JSON:
{{"opening_question": "one realistic opening interview question"}}
"""
    try:
        response = client.chat.completions.create(
            model=GROQ_MODEL,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.35,
            max_tokens=180,
        )
        parsed = safe_json_loads(response.choices[0].message.content, {})
        question = parsed.get("opening_question", "").strip()
    except Exception:
        question = ""
    if not question:
        question = f"Tell me about yourself and why you are interested in the {role} role."
    return {"opening_question": question}


def generate_final_report(history: List[Dict[str, Any]], role: str, persona: str, interview_mode: str) -> Dict[str, Any]:
    conversation = build_conversation(history)
    prompt = f"""
You are an expert interview coach.

Create a final interview report.
Role: {role}
Persona: {persona}
Mode: {interview_mode}

Conversation:
{conversation}

Return ONLY valid JSON:
{{
  "overall_score": 0,
  "hire_readiness": "not_ready|needs_practice|almost_ready|ready",
  "summary": "short summary",
  "top_strengths": ["...", "..."],
  "top_weaknesses": ["...", "..."],
  "communication_feedback": "...",
  "technical_feedback": "...",
  "recommended_practice_plan": ["...", "...", "..."],
  "next_7_day_plan": ["day 1 task", "day 2 task", "day 3 task"],
  "linkedin_project_summary": "one polished sentence describing this mock interview system"
}}
"""
    fallback = {
        "overall_score": 6,
        "hire_readiness": "needs_practice",
        "summary": "The candidate showed potential but needs more structured, specific, and result-focused answers.",
        "top_strengths": ["Completed the mock interview", "Communicated basic ideas"],
        "top_weaknesses": ["Needs stronger examples", "Needs clearer structure"],
        "communication_feedback": "Use concise STAR-format answers with fewer fillers.",
        "technical_feedback": "Add more implementation details and measurable outcomes.",
        "recommended_practice_plan": ["Prepare 3 project stories", "Practice STAR answers", "Review role-specific fundamentals"],
        "next_7_day_plan": ["Practice one introduction answer", "Prepare two project answers", "Record and review one mock interview"],
        "linkedin_project_summary": "Built an AI-powered mock interview assistant with speech transcription, adaptive questioning, and performance analytics.",
    }
    response = client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.35,
        max_tokens=1000,
    )
    parsed = safe_json_loads(response.choices[0].message.content, fallback)
    parsed["overall_score"] = int(clamp(int(parsed.get("overall_score", 6)), 0, 10))
    return parsed


# =========================
# ROUTES
# =========================

@app.route("/", methods=["GET"])
def home():
    return jsonify({
        "status": "running",
        "message": "AI Interview Backend is live",
        "endpoints": ["/health", "/opening_question", "/interview_turn", "/final_report", "/transcribe", "/ai_next", "/evaluate", "/confidence"],
    })


@app.route("/health", methods=["GET"])
def health():
    return jsonify({
        "ok": True,
        "whisper_model": WHISPER_MODEL_SIZE,
        "llm_model": GROQ_MODEL,
        "roles": list(ROLE_LIBRARY.keys()),
        "time": time.time(),
    })


@app.route("/opening_question", methods=["POST"])
def opening_question():
    try:
        data = request.json or {}
        return jsonify(generate_opening_question(
            role=data.get("role", "Software Engineer Intern"),
            persona=data.get("persona", "Friendly HR"),
            interview_mode=data.get("interview_mode", "Full Interview"),
            resume_context=data.get("resume_context", ""),
            job_description=data.get("job_description", ""),
        ))
    except Exception as exc:
        traceback.print_exc()
        return jsonify({"opening_question": "Tell me about yourself and your background.", "error": str(exc)}), 500


@app.route("/interview_turn", methods=["POST"])
def interview_turn():
    audio_path = None
    try:
        if "audio" not in request.files:
            return jsonify({"error": "Missing audio file. Send multipart field named audio."}), 400

        payload = json.loads(request.form.get("payload", "{}"))
        history = payload.get("history", [])
        persona = payload.get("persona", "Friendly HR")
        role = payload.get("role", "Software Engineer Intern")
        interview_mode = payload.get("interview_mode", "Full Interview")
        time_remaining_seconds = int(payload.get("time_remaining_seconds", 300))
        duration_seconds = float(payload.get("duration_seconds", 0))
        resume_context = payload.get("resume_context", "")
        job_description = payload.get("job_description", "")

        audio = request.files["audio"]
        audio_path = os.path.join(TEMP_DIR, f"interview_audio_{uuid.uuid4().hex}.wav")
        audio.save(audio_path)

        transcript = transcribe_audio_file(audio_path) or "No clear speech detected."
        speech_metrics = local_speech_score(transcript, duration_seconds)

        ai_result = ai_interview_turn(
            history=history,
            transcript=transcript,
            persona=persona,
            role=role,
            interview_mode=interview_mode,
            time_remaining_seconds=time_remaining_seconds,
            speech_metrics=speech_metrics,
            resume_context=resume_context,
            job_description=job_description,
        )

        return jsonify({
            "transcript": transcript,
            "speech_metrics": speech_metrics,
            **ai_result,
        })
    except Exception as exc:
        traceback.print_exc()
        return jsonify({
            "error": str(exc),
            "transcript": "",
            "score": 5,
            "feedback": "Server error while processing this turn.",
            "next_question": FALLBACK_QUESTIONS[int(time.time()) % len(FALLBACK_QUESTIONS)],
        }), 500
    finally:
        if audio_path and os.path.exists(audio_path):
            try:
                os.remove(audio_path)
            except Exception:
                pass


@app.route("/final_report", methods=["POST"])
def final_report():
    try:
        data = request.json or {}
        return jsonify(generate_final_report(
            history=data.get("history", []),
            role=data.get("role", "Software Engineer Intern"),
            persona=data.get("persona", "Friendly HR"),
            interview_mode=data.get("interview_mode", "Full Interview"),
        ))
    except Exception as exc:
        traceback.print_exc()
        return jsonify({"error": str(exc), "summary": "Could not generate final report."}), 500


# Legacy routes so older laptop code still works.
@app.route("/transcribe", methods=["POST"])
def transcribe():
    audio_path = None
    try:
        if "audio" not in request.files:
            return jsonify({"text": "", "error": "No audio file received"}), 400
        audio_path = os.path.join(TEMP_DIR, f"transcribe_{uuid.uuid4().hex}.wav")
        request.files["audio"].save(audio_path)
        return jsonify({"text": transcribe_audio_file(audio_path)})
    except Exception as exc:
        traceback.print_exc()
        return jsonify({"text": "", "error": str(exc)}), 500
    finally:
        if audio_path and os.path.exists(audio_path):
            try:
                os.remove(audio_path)
            except Exception:
                pass


@app.route("/ai_next", methods=["POST"])
def ai_next():
    try:
        data = request.json or {}
        history = data.get("history", [])
        latest_answer = history[-1].get("answer", "") if history else ""
        speech_metrics = local_speech_score(latest_answer, 0)
        ai_result = ai_interview_turn(
            history=history,
            transcript=latest_answer,
            persona=data.get("persona", "Friendly HR"),
            role=data.get("role", "Software Engineer Intern"),
            interview_mode=data.get("interview_mode", "Full Interview"),
            time_remaining_seconds=int(data.get("time_remaining_seconds", 300)),
            speech_metrics=speech_metrics,
        )
        return jsonify({"question": ai_result["next_question"], "next_question": ai_result["next_question"], **ai_result})
    except Exception as exc:
        traceback.print_exc()
        return jsonify({"question": FALLBACK_QUESTIONS[0], "error": str(exc)}), 500


@app.route("/evaluate", methods=["POST"])
def evaluate():
    try:
        data = request.json or {}
        answer = data.get("answer", "")
        question = data.get("question", "")
        ai_result = ai_interview_turn([], answer, "Friendly HR", "Software Engineer Intern", "Practice", 300, local_speech_score(answer, 0))
        return jsonify({"feedback": ai_result["feedback"], "score": ai_result["score"], "question": question, **ai_result})
    except Exception as exc:
        traceback.print_exc()
        return jsonify({"feedback": f"Error: {str(exc)}"}), 500


@app.route("/confidence", methods=["POST"])
def confidence():
    try:
        data = request.json or {}
        return jsonify(local_speech_score(data.get("answer", ""), float(data.get("duration_seconds", 0))))
    except Exception as exc:
        traceback.print_exc()
        return jsonify({"confidence": 50, "filler_count": 0, "error": str(exc)}), 500


# =========================
# START SERVER
# =========================

def run_flask():
    app.run(host="0.0.0.0", port=5000, threaded=True, use_reloader=False)


ngrok.set_auth_token(NGROK_AUTH_TOKEN)
try:
    ngrok.kill()
except Exception:
    pass

server_thread = threading.Thread(target=run_flask, daemon=True)
server_thread.start()
time.sleep(2)

public_url = ngrok.connect(5000)
print("Your public backend URL is:")
print(public_url)
print("Copy this URL into SERVER_URL in the laptop app.")
